In [3]:
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_columns", 20)

# Put the CSV next to the notebook, or change this path
DATA_PATH = r"D:\Mota\rock-n-roll-discover-server\processed\analytics_jobs_clean.csv"

## 1. Load

In [4]:
raw = pd.read_csv(DATA_PATH)
print(raw.shape)
raw.head()

(15841, 16)


,s_no,experience,job_description,job_desig,job_type,key_skills,location,salary,key_skills_normalized,job_text,experience_min,experience_max,experience_mid,salary_min,salary_max,salary_mid
0,1,6-10 yrs,Must be knowledgeable and experienced in relational data...,Oracle EBS Technical Consultant,NaN,"Oracle SQL, PLSQL, PL, Oracle Forms, Oracle Reports, Ora...","Bengaluru, Chennai",6to10,"oracle sql, plsql, pl, oracle forms, oracle reports, ora...",Role: Oracle EBS Technical Consultant. Description: Must...,NaN,NaN,NaN,NaN,NaN,NaN
1,2,8-12 yrs,- Experience in building scalable and highly available d...,Staff Software Engineer - Object Oriented Analysis & Design,NaN,"Javascript, HTML, JQuery, Web Development, Play Framewor...",Bengaluru,10to15,"javascript, html, jquery, web development, play framewor...",Role: Staff Software Engineer - Object Oriented Analysis...,NaN,NaN,NaN,NaN,NaN,NaN
2,3,3-8 yrs,NaN,Servicenow Developer,NaN,"SNOW, Servicenow, Servicenow Developer","Hyderabad, Pune",0to3,"snow, servicenow, servicenow developer","Role: Servicenow Developer. Required skills: snow, servi...",NaN,NaN,NaN,NaN,NaN,NaN
3,4,3-6 yrs,Our software engineers at Fiserv bring an open and creat...,"Software Development Engineering, Analyst",NaN,"QTP, Flex, LAN, Application support, Performance testing...",Pune,6to10,"qtp, flex, lan, application support, performance testing...","Role: Software Development Engineering, Analyst. Descrip...",NaN,NaN,NaN,NaN,NaN,NaN
4,5,5-10 yrs,- More than 5 years experience in Hypermesh for Finite E...,NVH Engineer - Nastran/ Ls-dyna,NaN,"HyperMesh, LS - DYNA, NASTRAN, Abaqus, Fatigue, Durabili...","Mumbai, Bengaluru, Hyderabad, Pune, Delhi",10to15,"hypermesh, ls - dyna, nastran, abaqus, fatigue, durabili...",Role: NVH Engineer - Nastran/ Ls-dyna. Description: - Mo...,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Clean
Drops the `...` skill artifact, spam-like listings and duplicates. Parses experience, salary band and city for filtering.

In [5]:
SALARY_ORDER = ["0to3", "3to6", "6to10", "10to15", "15to25", "25to50"]

df = raw.copy()
df["key_skills"] = df["key_skills"].fillna("")

# skill tokens: lowercase, drop the "..." scraping artifact
df["skills"] = df["key_skills"].apply(
    lambda s: [t.strip().lower() for t in s.split(",") if t.strip() and t.strip() != "..."]
)
df = df[df["skills"].str.len() > 0]

df["title"] = df["job_desig"].str.lower().str.strip()
df = df[~df["title"].str.contains("data entry|home base|part time work|freelanc")]
df = df.drop_duplicates(["title", "key_skills", "location", "experience", "salary"]).reset_index(drop=True)

# structured fields for filters
df[["exp_min", "exp_max"]] = df["experience"].str.extract(r"(\d+)-(\d+)").astype(float)
df["salary_idx"] = df["salary"].map({b: i for i, b in enumerate(SALARY_ORDER)})
df["cities"] = df["location"].str.lower().str.split(r"\s*,\s*")

# multi-word skills become ONE token, e.g. "machine learning" -> "machine_learning"
df["skill_txt"] = df["skills"].apply(lambda l: " ".join(t.replace(" ", "_") for t in l))
df["desc"] = df["job_description"].fillna("")

print(f"{len(raw)} -> {len(df)} rows after cleaning")

15841 -> 14729 rows after cleaning


## 3. Build the index
Weights are starting guesses. Tune them on queries you have judged yourself.

In [6]:
W_SKILLS, W_TITLE, W_DESC = 0.45, 0.30, 0.25

v_s = TfidfVectorizer(token_pattern=r"\S+", sublinear_tf=True, min_df=2)
v_t = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=2, stop_words="english")
v_d = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=3, max_features=50000, stop_words="english")

Xs = v_s.fit_transform(df["skill_txt"])
Xt = v_t.fit_transform(df["title"])
Xd = v_d.fit_transform(df["desc"])

def combine(s, t, d, ws=(W_SKILLS, W_TITLE, W_DESC)):
    parts = [s * np.sqrt(ws[0]), t * np.sqrt(ws[1]), d * np.sqrt(ws[2])]
    return normalize(sparse.hstack(parts).tocsr())

X = combine(Xs, Xt, Xd)
print("Index shape:", X.shape)

Index shape: (14729, 26953)


## 4. Search function
Text parts are scored by cosine similarity. Experience, city and salary are applied as filters after scoring.

In [7]:
def search(skills, title="", description="", exp_years=None, city=None, min_salary_band=None, k=10):
    qs = v_s.transform([" ".join(s.strip().lower().replace(" ", "_") for s in skills)])
    qt = v_t.transform([title.lower()])
    qd = v_d.transform([description])
    scores = (X @ combine(qs, qt, qd).T).toarray().ravel()

    mask = np.ones(len(df), dtype=bool)
    if exp_years is not None:
        mask &= (df["exp_min"] <= exp_years + 1) & (df["exp_max"] >= exp_years - 1)
    if city:
        mask &= df["cities"].apply(lambda c: any(city.lower() in x for x in c))
    if min_salary_band is not None:
        mask &= df["salary_idx"] >= SALARY_ORDER.index(min_salary_band)
    scores = np.where(mask, scores, -1)

    top = np.argsort(-scores)[:k]
    out = df.loc[top, ["job_desig", "key_skills", "experience", "salary", "location"]].copy()
    out["score"] = scores[top].round(3)
    return out[out["score"] > 0]

## 5. Try it

In [8]:
search(["python", "sql", "machine learning"], title="data scientist", exp_years=3, city="bengaluru", k=5)

,job_desig,key_skills,experience,salary,location,score
12276,Data Scientist,"Machine Learning, Python, R, Data Visualization, Data Mi...",3-8 yrs,15to25,Bengaluru,0.627
5519,Data Scientist,"R, Machine Learning, Customer Analytics, Python, Data An...",1-5 yrs,6to10,Bengaluru,0.559
6148,Data Scientist,"Deep Learning, Machine Learning, Python, R, Artificial I...",2-4 yrs,15to25,Bengaluru,0.555
14555,Data Scientist,"R, C, Machine Learning, Data Analytics, Python, Deep Lea...",3-7 yrs,6to10,Bengaluru,0.535
2224,DATA Scientist,"Python, Analytics, Data analysis, Data mining, Machine l...",2-7 yrs,3to6,Bengaluru,0.530


In [9]:
search(["sql", "excel", "tableau"], title="data analyst", exp_years=1, k=5)

,job_desig,key_skills,experience,salary,location,score
13861,Data Analyst,"Tableau, Web Analytics, Google Analytics, SQL, Reporting...",1-3 yrs,10to15,Mumbai,0.554
2523,Data Analyst,"Python, SAS, R, SQL, Analytics, Excel, Statistical Model...",2-7 yrs,10to15,Mumbai,0.536
3421,Data Analyst,"excel, data analysis, statistics, customer centric, Pyth...",1-5 yrs,10to15,Noida,0.517
9313,Data Analyst,"excel, hlookup, macros, data analysis, VLOOKUP, pivot table",1-3 yrs,0to3,Hyderabad(Kondapur),0.502
13832,Data Analyst,"SQL, ETL Tool, Informatica, Talend, Excel, Data Analysis...",2-6 yrs,0to3,Hyderabad,0.497


## 6. Find jobs similar to an existing posting

In [10]:
def similar_to(row_idx, k=5):
    scores = (X @ X[row_idx].T).toarray().ravel()
    scores[row_idx] = -1                      # exclude itself
    top = np.argsort(-scores)[:k]
    out = df.loc[top, ["job_desig", "key_skills", "experience", "salary", "location"]].copy()
    out["score"] = scores[top].round(3)
    return out

print("Query posting:", df.loc[100, "job_desig"], "|", df.loc[100, "key_skills"][:80])
similar_to(100)

Query posting: Data Modeler - Fsldm | multivariate analysis, etl, data warehousing, data modeling...


,job_desig,key_skills,experience,salary,location,score
7613,Data Modeler,"Multivariate Analysis, Time Series Analysis, Cluster Ana...",5-8 yrs,15to25,Trivandrum,0.467
8735,Data Modeler,"SQL, Python, Data modeling, Database design, Data analys...",10-15 yrs,15to25,Pune,0.303
12701,Talend Developer - Etl/data Warehouse/business Intelligence,"Data Integration, SQL Queries, ETL, Data Warehousing, Da...",4-9 yrs,15to25,Bengaluru,0.292
6088,Senior Data Engineer - Cloud BI Production Implementation,"Power Bi, SSIS, MS SQL Server, ETL, Data Warehousing, Da...",4-8 yrs,6to10,Chennai,0.266
3853,Senior Data Engineer - Cloud BI Production Implementation,"Power Bi, SSIS, MS SQL Server, ETL, Data Warehousing, Da...",4-8 yrs,10to15,Chennai,0.266


## 7. Sanity check (weak proxy, not real accuracy)
For random postings, how often are the top-10 neighbours in the same title family?

**Caution:** the family label comes from the title, so *title-only* scores high by construction. The **skills-only** number is the more meaningful one. Real evaluation needs queries whose results you have judged by hand.

In [11]:
def family(t):
    for k, c in [("data scien", "ds"), ("machine learning", "ds"), ("data analy", "da"),
                 ("business analy", "ba"), ("seo", "mkt"), ("digital marketing", "mkt"),
                 ("product manager", "pm"), ("data engineer", "de")]:
        if k in t:
            return c
    return None

df["family"] = df["title"].apply(family)
labeled = df.index[df["family"].notna()]
q = np.random.default_rng(0).choice(labeled, 400, replace=False)

def weak_precision(ws):
    Xc = combine(Xs, Xt, Xd, ws)
    S = (Xc[q] @ Xc.T).toarray()
    S[np.arange(len(q)), q] = -1
    top = np.argsort(-S, axis=1)[:, :10]
    vals = []
    for i, qi in enumerate(q):
        fam = df["family"].values[top[i]]
        m = pd.notna(fam)
        if m.any():
            vals.append((fam[m] == df["family"].values[qi]).mean())
    return round(float(np.mean(vals)), 3)

for name, ws in [("skills only", (1, 0, 0)), ("title only (circular)", (0, 1, 0)),
                 ("description only", (0, 0, 1)), ("skills + title", (0.6, 0.4, 0)),
                 ("skills + title + desc", (W_SKILLS, W_TITLE, W_DESC))]:
    print(f"{name:25s} {weak_precision(ws)}")

skills only               0.822
title only (circular)     0.948
description only          0.692
skills + title            0.937
skills + title + desc     0.933


In [12]:
import os
import re
import json
import pickle
import ast
import numpy as np
import pandas as pd

from pathlib import Path
from sentence_transformers import SentenceTransformer

print("Libraries loaded.")


Libraries loaded.


In [19]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path(
    r"D:\Mota\rock-n-roll-discover-server\processed\analytics_jobs_clean.csv"
)

print("Using:", DATA_PATH)
print("Exists:", DATA_PATH.exists())

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Using: D:\Mota\rock-n-roll-discover-server\processed\analytics_jobs_clean.csv
Exists: True
Shape: (15841, 16)

Columns:
['s_no', 'experience', 'job_description', 'job_desig', 'job_type', 'key_skills', 'location', 'salary', 'key_skills_normalized', 'job_text', 'experience_min', 'experience_max', 'experience_mid', 'salary_min', 'salary_max', 'salary_mid']


In [20]:
required_columns = [
    "job_desig",
    "job_description",
    "key_skills_normalized",
    "experience_min",
    "experience_max",
    "job_text"
]

for col in required_columns:
    print(f"{col:30} -> {col in df.columns}")

job_desig                      -> True
job_description                -> True
key_skills_normalized          -> True
experience_min                 -> True
experience_max                 -> True
job_text                       -> True


In [21]:
import re
import ast
import numpy as np

def normalize_skill_set(skills):
    if skills is None:
        return set()

    if isinstance(skills, float) and np.isnan(skills):
        return set()

    if isinstance(skills, np.ndarray):
        skills = skills.tolist()

    if isinstance(skills, (list, tuple, set)):
        items = skills
    else:
        text = str(skills).strip()

        if not text or text.lower() in {"nan", "none", "null"}:
            return set()

        if text.startswith("[") and text.endswith("]"):
            try:
                parsed = ast.literal_eval(text)
                items = parsed if isinstance(parsed, (list, tuple, set)) else [text]
            except Exception:
                items = re.split(r"[,;|]", text)
        else:
            items = re.split(r"[,;|]", text)

    return {
        str(x).strip().lower()
        for x in items
        if x is not None and str(x).strip().lower() not in {"", "nan", "none"}
    }

In [22]:
df["skill_set"] = df["key_skills_normalized"].apply(normalize_skill_set)

print(df["skill_set"].head())

0    {oracle sql, plsql, oracle forms, oracle reports, pl, or...
1    {html, play framework, mvc, web development, rest..., jq...
2                       {servicenow developer, snow, servicenow}
3    {qtp, eclipse..., application support, lan, performance ...
4    {ls - dyna, nvh, abaqus, fea..., hypermesh, durability, ...
Name: skill_set, dtype: object


In [23]:
df["job_text"] = df["job_text"].fillna("").astype(str)

df = df[df["job_text"].str.len() >= 20].copy()

df.reset_index(drop=True, inplace=True)

print("Usable jobs:", len(df))

Usable jobs: 15841


In [24]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(MODEL_NAME)

print("Loaded:", MODEL_NAME)

Loaded: sentence-transformers/all-MiniLM-L6-v2


In [25]:
job_texts = df["job_text"].tolist()

job_embeddings = embedding_model.encode(
    job_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

job_embeddings = np.asarray(job_embeddings, dtype="float32")

print("Embedding shape:", job_embeddings.shape)

Batches:   0%|          | 0/248 [00:00<?, ?it/s]

Embedding shape: (15841, 384)


In [26]:
import faiss

dimension = job_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(job_embeddings)

print("FAISS index created")
print("Vectors:", index.ntotal)
print("Dimension:", dimension)

FAISS index created
Vectors: 15841
Dimension: 384


In [27]:
def safe_float(value):
    try:
        if pd.isna(value):
            return None
        return float(value)
    except:
        return None


def experience_score(candidate_exp, min_exp, max_exp):

    candidate_exp = safe_float(candidate_exp)
    min_exp = safe_float(min_exp)
    max_exp = safe_float(max_exp)

    if candidate_exp is None:
        return 0.5

    if min_exp is None and max_exp is None:
        return 0.5

    if min_exp is not None and max_exp is None:
        if candidate_exp >= min_exp:
            return 1.0
        return max(0, 1 - (min_exp - candidate_exp) / 5)

    if min_exp is None and max_exp is not None:
        if candidate_exp <= max_exp:
            return 1.0
        return max(0, 1 - (candidate_exp - max_exp) / 5)

    if min_exp <= candidate_exp <= max_exp:
        return 1.0

    gap = (
        min_exp - candidate_exp
        if candidate_exp < min_exp
        else candidate_exp - max_exp
    )

    return max(0, 1 - gap / 5)

In [28]:
def skill_score(candidate_skills, job_skills):

    candidate_skills = normalize_skill_set(candidate_skills)
    job_skills = normalize_skill_set(job_skills)

    if not job_skills:
        return 0.5

    if not candidate_skills:
        return 0.0

    matched = candidate_skills.intersection(job_skills)

    return len(matched) / len(job_skills)

In [29]:
STOPWORDS = {
    "the", "and", "for", "with", "of", "to",
    "a", "an", "in", "on", "at", "is", "are",
    "engineer", "developer", "analyst", "manager"
}


def tokenize_role(text):

    tokens = re.findall(
        r"[a-zA-Z0-9]+",
        str(text).lower()
    )

    return {
        token
        for token in tokens
        if token not in STOPWORDS and len(token) > 2
    }


def role_score(candidate_title, job_title):

    candidate_tokens = tokenize_role(candidate_title)
    job_tokens = tokenize_role(job_title)

    if not candidate_tokens or not job_tokens:
        return 0.5

    intersection = candidate_tokens.intersection(job_tokens)

    return len(intersection) / len(candidate_tokens)

In [30]:
def semantic_retrieve(candidate_text, top_k=30):

    candidate_embedding = embedding_model.encode(
        [candidate_text],
        normalize_embeddings=True
    )

    candidate_embedding = np.asarray(
        candidate_embedding,
        dtype="float32"
    )

    scores, indices = index.search(
        candidate_embedding,
        top_k
    )

    return scores[0], indices[0]

In [31]:
def hybrid_match(
    candidate_title,
    candidate_skills,
    candidate_experience,
    candidate_description="",
    retrieve_k=30,
    final_k=10
):

    candidate_skills = normalize_skill_set(candidate_skills)

    candidate_text = f"""
    Job Role: {candidate_title}

    Skills: {", ".join(sorted(candidate_skills))}

    Experience: {candidate_experience} years

    Description:
    {candidate_description}
    """

    semantic_scores, indices = semantic_retrieve(
        candidate_text,
        top_k=retrieve_k
    )

    results = []

    for semantic_score, idx in zip(
        semantic_scores,
        indices
    ):

        job = df.iloc[int(idx)]

        skills = skill_score(
            candidate_skills,
            job["skill_set"]
        )

        experience = experience_score(
            candidate_experience,
            job["experience_min"],
            job["experience_max"]
        )

        role = role_score(
            candidate_title,
            job["job_desig"]
        )

        final_score = (
            0.55 * float(semantic_score)
            + 0.25 * skills
            + 0.10 * experience
            + 0.10 * role
        )

        results.append({
            "job_index": int(idx),
            "job_title": job["job_desig"],
            "location": job.get("location", ""),
            "experience_required": job.get("experience", ""),
            "semantic_score": float(semantic_score),
            "skill_score": float(skills),
            "experience_score": float(experience),
            "role_score": float(role),
            "final_score": float(final_score)
        })

    result_df = pd.DataFrame(results)

    result_df = (
        result_df
        .sort_values("final_score", ascending=False)
        .head(final_k)
        .reset_index(drop=True)
    )

    result_df.insert(
        0,
        "rank",
        range(1, len(result_df) + 1)
    )

    return result_df

In [32]:
candidate = {
    "title": "Data Scientist",

    "skills": [
        "Python",
        "SQL",
        "Machine Learning",
        "Pandas",
        "TensorFlow",
        "Statistics"
    ],

    "experience": 2,

    "description": """
    Data Scientist with experience in Python,
    machine learning, data analysis,
    predictive modeling and statistics.
    """
}

In [33]:
results = hybrid_match(
    candidate_title=candidate["title"],
    candidate_skills=candidate["skills"],
    candidate_experience=candidate["experience"],
    candidate_description=candidate["description"],
    retrieve_k=30,
    final_k=10
)

results

,rank,job_index,job_title,location,experience_required,semantic_score,skill_score,experience_score,role_score,final_score
0,1,15345,Data Scientist - Fintech Startup,"Gurgaon, Gurugram",3-5 yrs,0.743104,0.500000,0.5,1.0,0.683707
1,2,3497,Job Opening for Business Analytics /data Scientist,Bengaluru,5-9 yrs,0.746779,0.428571,0.5,1.0,0.667872
2,3,12322,Job Opening for Business Analytics /data Scientist,Bengaluru,5-9 yrs,0.746779,0.428571,0.5,1.0,0.667872
3,4,10168,Job Opening for Business Analytics /data Scientist,Bengaluru,5-9 yrs,0.746779,0.428571,0.5,1.0,0.667872
4,5,4524,Data Scientist,Noida(Sector-126 Noida),5-7 yrs,0.758144,0.400000,0.5,1.0,0.666979
5,6,419,Data Scientist,Noida(Sector-126 Noida),5-7 yrs,0.758144,0.400000,0.5,1.0,0.666979
6,7,13017,Data Scientist,Pune(Viman Nagar),1-5 yrs,0.743914,0.428571,0.5,1.0,0.666295
7,8,3514,Data Analyst,Noida,1-5 yrs,0.785711,0.500000,0.5,0.5,0.657141
8,9,15785,Data Scientist - Python/nlp/aws,Mumbai,2-7 yrs,0.733713,0.400000,0.5,1.0,0.653542
9,10,4968,Data Scientist / Engineer - R/python/sql,Noida,2-5 yrs,0.745840,0.333333,0.5,1.0,0.643545


In [34]:
top = results.iloc[0]

print("TOP MATCH")
print("=" * 50)

print("Job:", top["job_title"])
print("Location:", top["location"])

print("\nScore Breakdown")
print("-" * 50)

print(f"Semantic similarity : {top['semantic_score']:.3f}")
print(f"Skill compatibility : {top['skill_score']:.3f}")
print(f"Experience match    : {top['experience_score']:.3f}")
print(f"Role compatibility  : {top['role_score']:.3f}")

print("-" * 50)

print(f"FINAL HYBRID SCORE  : {top['final_score']:.3f}")

TOP MATCH
Job: Data Scientist - Fintech Startup
Location: Gurgaon, Gurugram

Score Breakdown
--------------------------------------------------
Semantic similarity : 0.743
Skill compatibility : 0.500
Experience match    : 0.500
Role compatibility  : 1.000
--------------------------------------------------
FINAL HYBRID SCORE  : 0.684


In [35]:
from pathlib import Path
import pickle
import json

MODEL_DIR = Path(
    r"D:\Mota\rock-n-roll-discover-server\models\semantic_hybrid"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(MODEL_DIR)

D:\Mota\rock-n-roll-discover-server\models\semantic_hybrid


In [36]:
np.save(
    MODEL_DIR / "job_embeddings.npy",
    job_embeddings
)

faiss.write_index(
    index,
    str(MODEL_DIR / "jobs.faiss")
)

metadata = df.drop(
    columns=["skill_set"],
    errors="ignore"
)

with open(
    MODEL_DIR / "job_metadata.pkl",
    "wb"
) as f:
    pickle.dump(metadata, f)

config = {
    "embedding_model": MODEL_NAME,
    "semantic_weight": 0.55,
    "skill_weight": 0.25,
    "experience_weight": 0.10,
    "role_weight": 0.10,
    "retrieval_k": 30,
    "final_k": 10,
    "embedding_dimension": int(dimension)
}

with open(
    MODEL_DIR / "config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(config, f, indent=4)

results.to_csv(
    MODEL_DIR / "sample_hybrid_results.csv",
    index=False
)

print("Model saved successfully.")

Model saved successfully.


In [37]:
for file in MODEL_DIR.iterdir():
    print(file.name)

config.json
jobs.faiss
job_embeddings.npy
job_metadata.pkl
sample_hybrid_results.csv
